# 02 · 去噪模型训练（Colab）

训练 `crn-nano` / `crn-lite`。V1 的目标是**单通道实时去噪**，不是去混响：
加 RIR 时，训练目标仍是带混响但无加性噪声的 `wet` 语音。

训练使用 DNS5 干净语音、真实噪声和真实 RIR，并在线随机混音。当前配方显式覆盖
高 SNR 与 clean→clean 恒等样本，避免旧模型对干净输入无条件过抑制。

先用 `SMOKE_RUN=True` 跑 `crn-nano` 3 epoch 验证数据、训练、断点与导出；该结果
只证明链路可运行，不用于报告质量。正式训练再切到 False。

产物保存到 Drive：`last.pt` 用于续训，`best.pt` 用于导出，`history.json` 保存曲线。


In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# V1 已停用 QUICK_TEST：两套受控集都依赖 DNS 留出噪声/RIR。
# 小规模跑通请用 SMOKE_RUN=True；QUICK_TEST 必须保持 False。
QUICK_TEST = False

# ── 小规模跑通模式 ─────────────────────────────────────────────────────
# True = 大幅缩小**用量**与**训练时长**，验证「数据→训练→导出→评测」整条链路。
# 缩的不是下载量（DNS 分片是最小单位，该下多少还是多少），而是"用多少条"和"跑多少轮"：
#   受控集   81 格 × 1 = 81 条/套（正式 5/格 = 405）
#   真实 CER 3 个时长桶 × 10 = 30 条（正式 300）
#   噪声分类 抽 400 条做平稳性判决（正式 4000）
#   训练     2000 样本/epoch × 3 epoch，只跑 crn-nano（正式 20000 × 60，两档）
# 跑通之后改成 False 再跑正式版。
SMOKE_RUN = True

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'
assert not QUICK_TEST, 'V1 不支持 QUICK_TEST；请改用 SMOKE_RUN=True 做小规模闭环'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSETS_DIR = f'{DRIVE}/testsets'  # V1 三套职责分离的固定测试集
DNS_QUALITY_DIR = f'{TESTSETS_DIR}/dns_objective'
AISHELL_CER_DIR = f'{TESTSETS_DIR}/aishell_controlled'
WENET_REAL_DIR = f'{TESTSETS_DIR}/wenetspeech_real'
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSETS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  三套固定测试集      {TESTSETS_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 训练/质量 + AISHELL 受控 CER + WenetSpeech 真实 CER)"}')
print(f'  规模      {"⚡ 小规模跑通（SMOKE_RUN=True，结果不作数）" if SMOKE_RUN else "正式规模"}')
print()

!df -h /content | tail -1


In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
# **重新导入前必须把已加载的 rtse 从 sys.modules 里清掉。**
# 上面的 unzip 换的是磁盘上的文件，而 `import rtse` 对**已经导入过**的模块是空操作：
# 同一个 Colab 会话里重跑本 cell，磁盘上是新代码、内存里跑的还是旧的。
# 这个症状极具迷惑性——报错的行号来自旧文件，跟你手里的新文件对不上号，
# 会让人以为"包没传上去"而反复重传（见 docs/ISSUES.md I-13 / I-28）。
for _m in [m for m in list(sys.modules) if m == 'rtse' or m.startswith('rtse.')]:
    del sys.modules[_m]
import rtse
# 自证：把**实际加载的文件路径和改动时间**打出来。
# "我改的代码到底有没有在跑"必须是可观测的事实，不能靠推断（I-28）。
print('已加载 rtse ←', rtse.__file__)
print('           改动时间',
      time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(rtse.__file__))),
      '| 若这个时间不是你刚打包的时刻，说明跑的还是旧代码')
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。Colab 上的 STFT 与本地哪怕差一点，训练出来的模型拿回本地就会
# 掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 构建数据集

Colab 每次连接给的都是**全新虚拟机**，本地盘是空的 —— 哪怕几分钟前刚在 01 里
跑完下载。下面会用 Drive 上缓存的压缩包重新解压（几分钟），压缩包也不在了才重下。
**这是正常现象，不用手动跳回 01。**

In [ ]:
mf = Path(f'{DRIVE}/manifest.json')
assert mf.exists(), f'找不到 {mf}，先跑 01_data_prep.ipynb'
manifest = json.loads(mf.read_text(encoding='utf-8'))
QUICK_TEST = manifest['quick_test']
print(f'清单版本: {manifest.get("version")}   quick_test={QUICK_TEST}')
print(f'  语音 train/val: {len(manifest["speech"]["train"])}/{len(manifest["speech"]["val"])}')
print(f'  噪声 稳态/非稳态(train): {len(manifest["noise_stationary_train"])}/'
      f'{len(manifest["noise_nonstationary_train"])}')
print(f'  真实 RIR train: {len(manifest["rir_train"])}')

In [ ]:
DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

# lbzip2：多线程解压 bzip2。噪声分片是 .tar.bz2，单线程解 5 GB 要 ~9 分钟，
# 而 Colab 每个会话都得重解一遍（/content 会被清空），这是最大的一块固定开销。
# 装不上也能跑，只是退回单线程。
!apt-get -qq install -y lbzip2 > /dev/null 2>&1 || true
print('lbzip2:', '可用（解压将并行）' if shutil.which('lbzip2') else '不可用（退回单线程 bzip2）')

def fetch_dns(name, blob_path, expect_min_wavs=50, partial_ok=False):
    """下载 → 解压一个 DNS 分片。带下载/解压双标记，支持断点续传与跨会话复用。

    Args:
        partial_ok: 该分片是 `split` 切片（干净语音），解压到末尾必然报
            "Unexpected EOF"。设 True 时忽略这个错误——只要解出足够多的
            完整文件就算成功。校验靠**实际解出的 wav 数量**，不靠 tar 的返回码。

    校验方式统一是"解压后递归扫到的 wav 数量"，而不是断言某个具体子目录名——
    实测 DNS5 语音解出来的路径是 `mnt/dnsv5/clean/read_speech/...`，
    嵌套好几层且没写在官方文档里。下游 scan() 本来就是递归扫描，不关心层数。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    # 解压标记里记下**当时解出的文件数**和格式版本。只有"新格式 + 文件数对得上"
    # 才认为可信。旧格式（空文件 touch 出来的）可能是 I-26 修复前写下的——
    # 那时截断的解压也会被当成成功（实测有分片只解出 1156/7739 却被记成已完成），
    # 一律重解一遍。这样**修复能追溯地清理掉此前留下的坏状态**，
    # 而不是只对将来生效、让已经写坏的标记永远把分片挡在门外。
    if os.path.exists(ex_mark):
        try:
            mark = json.loads(Path(ex_mark).read_text())
        except Exception:
            mark = None
        n_now = count_wavs()
        if isinstance(mark, dict) and mark.get('v', 0) >= 2 and mark.get('n') == n_now:
            print(f'[skip  ] {name} 已解压（{n_now} 个 wav）'); return True
        why = ('标记是旧格式，无法确认当时是否解压完整'
               if mark is None else
               f'文件数与标记不符（记录 {mark.get("n")}，实际 {n_now}）')
        print(f'[recheck] {name} {why} —— 重解一遍')

    def free_gb(path):
        try:
            return shutil.disk_usage(path).free / 1e9
        except Exception:
            return float('nan')

    # 下载标记里记下**当时的字节数**，缓存命中时核对一遍。
    # 只看"标记在不在"是不够的：压缩包可能被截断、被覆盖、或下到一半留下残file。
    # 旧格式（`touch` 出来的空文件）没有这个信息，就地升级成新格式，
    # **不重新下载**——那是 20 GB，代价太大，而且现有包已逐个核对过与服务器一致。
    cached_ok = False
    if os.path.exists(dl_mark) and os.path.exists(archive):
        size_now = os.path.getsize(archive)
        try:
            rec = json.loads(Path(dl_mark).read_text())
        except Exception:
            rec = None
        if isinstance(rec, dict) and rec.get('bytes') is not None:
            if rec['bytes'] == size_now:
                cached_ok = True
                print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，大小与记录一致）')
            else:
                # 用**字节数**报差异，不用 GB：两个只差几 MB 的数字，
                # 格式化成 GB 会显示成一模一样，等于没给信息。
                print(f'[FAIL  ] {name} 压缩包大小与记录不符 —— 文件被改动或截断。')
                print(f'         记录 {rec["bytes"]:,} 字节，实际 {size_now:,} 字节'
                      f'（差 {size_now - rec["bytes"]:+,}）')
                print(f'         删掉 {archive} 和 {dl_mark} 后重跑，或直接重跑让 -c 续传。')
                return False
        else:
            cached_ok = True
            Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size_now}))
            print(f'[cached] {name} 压缩包已在 Drive（{size_now/1e9:.2f} GB，标记已补记大小）')
    if not cached_ok:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        wlog = f'{ARCHIVE_DIR}/.{name}.wget.log'
        # 进度**必须可见**：几个 GB 的下载如果一声不吭，中途看起来和卡死没区别。
        # 同时又要留下日志，失败时才说得出原因，所以用 tee 兼顾两者。
        # `-T 60` 的超时在长连接上很容易触发，但 `-c` 会断点续传——
        # 重跑本 cell 就能接着下，**不要删掉已下载的部分**。
        # 走 bash 是因为要取 PIPESTATUS（Colab 的 /bin/sh 是 dash，不支持）。
        cmd = (f'wget --progress=dot:giga -c -T 60 -O {shq(archive)} {shq(url)} '
               f'2>&1 | tee {shq(wlog)}; exit ${{PIPESTATUS[0]}}')
        rc = subprocess.run(['bash', '-c', cmd]).returncode
        size = os.path.getsize(archive) if os.path.exists(archive) else 0
        if rc != 0 or size < 1e6:
            # **不要直接断言"blob 路径变了"**——那只是众多可能之一，而且是最不可能的那个。
            # 实测最常见的是**空间不足**：hybrid 模式下归档写在 Drive 上，
            # 本批归档合计约 20 GB，而 Drive 免费版只有 15 GB。
            # 先把证据摆出来（wget 原话 + 两个卷的剩余空间），再让人去判断。
            print(f'[FAIL  ] {name} 下载未完成（wget 退出码 {rc}，已落盘 {size/1e9:.2f} GB）')
            print('         ⚠️ 多数情况下这只是超时中断，**已下载的部分是有效的**：'
                  '直接重跑本 cell，-c 会接着下。')
            tail = subprocess.run(f'tail -n 3 {shq(wlog)}', shell=True,
                                  capture_output=True, text=True).stdout.strip()
            if tail:
                print('         wget: ' + tail.replace(chr(10), chr(10) + '         '))
            print(f'         剩余空间：归档卷 {free_gb(ARCHIVE_DIR):.1f} GB，'
                  f'解压卷 {free_gb(DATA):.1f} GB')
            print('         本批归档合计约需 20 GB（语音 5.2 + 噪声 14.2 + IR 0.3）。')
            print('         排查顺序：① 上面两个卷的剩余空间够不够；'
                  '② Drive 挂载是否还活着（ls 一下 DRIVE 目录）；')
            print('         ③ 都正常再去 https://github.com/microsoft/DNS-Challenge '
                  '核对 blob 路径（这一项本地 HEAD 实测过 200，最不可能）。')
            return False
        Path(dl_mark).write_text(json.dumps({'v': 2, 'bytes': size}))

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    # 压缩格式**按文件头判断，不能看扩展名**：切片档叫 `read_speech.tgz.partaa`，
    # 结尾是 `.partaa` 而不是 `.tgz`，用 endswith 会判成 bzip2；GNU tar 拿 -j 去解
    # gzip 流会直接报 "is not a bzip2 file" 且一个文件都不解出。
    # Colab 上实际踩过：语音分片解出 0 个 wav（见 docs/ISSUES.md I-25）。
    with open(archive, 'rb') as fh:
        magic = fh.read(2)
    if magic == b'\x1f\x8b':
        flag, prog = 'xzf', None
    else:
        # **bzip2 解压是单线程 CPU 瓶颈**：5 GB 分片实测约 9 分钟，全程只吃一个核。
        # lbzip2 能对**任意** bzip2 流做多线程解压（pbzip2 只能并行它自己压出来的），
        # Colab 两个 vCPU 大致能快一倍。装不上就退回单线程，不影响正确性。
        prog = 'lbzip2' if shutil.which('lbzip2') else None
        flag = 'xf' if prog else 'xjf'
    decomp = f'--use-compress-program={prog} ' if prog else ''
    # 切片档忽略 tar 的非零返回码（末尾必然 EOF），靠文件数判断成败。
    # stderr 落盘而不是丢进 /dev/null——失败时要能说出为什么失败。
    log = f'{out_dir}/.untar.log'
    with open(log, 'w') as lf:
        rc_tar = subprocess.run(f'tar {decomp}-{flag} {shq(archive)} -C {shq(out_dir)}',
                                shell=True, stderr=lf).returncode
    n = count_wavs()

    def tar_err():
        t = subprocess.run(f'tail -n 5 {shq(log)}', shell=True,
                           capture_output=True, text=True).stdout.strip()
        return ('\n         tar: ' + t.replace(chr(10), chr(10) + '         ')) if t else ''

    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav（用 {flag} 解，文件头 {magic!r}）。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB')
        print(f'         删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False
    # 切片档解到末尾必然 EOF，退出码非零属正常；**其余压缩包退出码非零意味着解压被截断**，
    # 而此时 wav 数很可能仍然过线，于是被当成成功放过去。
    # 实测踩过：噪声分片只解出 1156/7739 个文件却报 [done]，训练数据悄悄少了 85%。
    if rc_tar != 0 and not partial_ok:
        print(f'[FAIL  ] {name} 解压未完成：tar 退出码 {rc_tar}，只解出 {n} 个 wav。' + tar_err())
        print(f'         剩余空间：解压卷 {free_gb(DATA):.1f} GB —— 空间不足是最常见原因。')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive); Path(dl_mark).unlink(missing_ok=True)
    # 记下文件数，下次才能判断"这份解压结果还是不是完整的那一份"
    Path(ex_mark).write_text(json.dumps({'v': 2, 'n': n}))
    note = '（切片档，尾部 EOF 属正常）' if partial_ok else ''
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒 {note}')
    return True


def dns_shards():
    """按配置生成要下载的分片清单 (名字, blob 路径)。"""
    # 语音是 split 切片，按字母序 partaa/partab/...；每片约 5.24 GB ≈ 19 小时
    parts = ['aa', 'ab', 'ac', 'ad', 'ae']
    sp = [(f'dns_speech_{p}', f'Track1_Headset/read_speech.tgz.part{p}')
          for p in parts[:N_SPEECH_SHARDS]]
    nz = [(f'dns_noise_audioset_{i:03d}',
           f'noise_fullband/datasets_fullband.noise_fullband.audioset_{i:03d}.tar.bz2')
          for i in range(N_AUDIOSET_SHARDS)]
    nz += [(f'dns_noise_freesound_{i:03d}',
            f'noise_fullband/datasets_fullband.noise_fullband.freesound_{i:03d}.tar.bz2')
           for i in range(N_FREESOUND_SHARDS)]
    # ⚠️ IR 分片在 blob 根目录下，**没有** `impulse_responses/` 前缀
    # （语音和噪声分片才有目录前缀）。写错会 404 —— 本地 HEAD 请求实测确认过。
    ir = [('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2')]
    return sp, nz, ir

# 语料解压在临时盘，新会话要重来一遍。复用 01 里同一个 fetch_dns()。
sp_shards, nz_shards, ir_shards = dns_shards()
if not QUICK_TEST:
    ok = {}
    for n, b in sp_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=500, partial_ok=True)
    for n, b in nz_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=100)
    for n, b in ir_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=50)
    assert all(ok.values()), '有语料没能自动补齐，看上面的 FAIL 信息'
print('语料就绪。')

In [ ]:
from torch.utils.data import DataLoader
from rtse.data.dataset import OnlineMixDataset, MixConfig
from rtse.metrics.intrusive import si_sdr

# OnlineMixDataset 接受**文件列表**。必须用清单而不是让它扫目录 ——
# 清单是按说话人划分好的，扫目录会把验证说话人混进训练集，指标全部虚高。
# 用 MixConfig 的**默认值**，不要在这里写死 snr_range ——
# 默认值刚修过（上限 20→35 dB、恒等样本 2%→10%）：原来模型从没见过
# "已经干净"的输入，学成了无条件过抑制，下游 CER 在高 SNR 段大幅恶化。
# 在这里覆盖默认值 = 把那个 bug 请回来（见 docs/FINDINGS.md F-11）。
MIX = MixConfig(segment_seconds=4.0)

train_ds = OnlineMixDataset(manifest['speech']['train'], manifest['noise_train'],
                            manifest['rir_train'], cfg=MIX, length=2000 if SMOKE_RUN else 20000, seed=0)
val_ds = OnlineMixDataset(manifest['speech']['val'], manifest['noise_test'],
                          manifest['rir_test'], cfg=MIX, length=200 if SMOKE_RUN else 800, seed=999)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

nb_, cl_ = next(iter(train_dl))
print('batch', tuple(nb_.shape), '| 输入 SI-SDR 抽样:',
      [round(si_sdr(cl_[i].numpy(), nb_[i].numpy()), 1) for i in range(4)], 'dB')

## 2. 训练

**别照搬预估时间，看第一个 epoch 的实测值。** 训练器把每个 epoch 的
`epoch_seconds` 写进 `history.json`，第一个 epoch 跑完就能算出总时长。

第一次建议把 `EPOCHS` 改成 5 跑一轮，确认 loss 在降、时间可接受，再改回 60。
先跑 `crn-nano`（参数量只有 lite 的 1/5），它跑完就能拿到一套完整的端到端指标，
把整个流程闭环；大模型再慢慢跑。**有一个能用的模型，远好过两个都卡在半路。**

In [ ]:
from rtse.models import build_model
from rtse.train import Trainer, TrainConfig

MODELS = ['crn-nano'] if SMOKE_RUN else ['crn-nano', 'crn-lite']
EPOCHS = 3 if SMOKE_RUN else 60

for name in MODELS:
    out_dir = f'{CKPT_DIR}/{name}'     # ← 落在 Drive 上，会话断了也在
    os.makedirs(out_dir, exist_ok=True)
    cfg = TrainConfig(model=name, epochs=EPOCHS, batch_size=16, lr=3e-4,
                      out_dir=out_dir, num_workers=2, log_every=100)
    model = build_model(name)
    print(f'\n{"="*70}\n{name}   参数量 {model.count_params():,}   → {out_dir}\n{"="*70}')

    tr = Trainer(model, train_dl, val_dl, cfg)
    last = Path(out_dir) / 'last.pt'
    if last.exists():
        tr.load(last)                  # 断点续训（含优化器/调度/随机数状态）
    if tr.epoch >= EPOCHS:
        print(f'{name} 已完成（epoch {tr.epoch}），跳过'); continue
    tr.fit()

print('\n训练产物（在 Drive 上）：')
!ls -lh {shq(CKPT_DIR)}/*/

## 3. 无害性与有效性闸门

训练loss下降不等于模型可部署。导出前同时检查：

- clean→clean：完全干净输入不应再次被重度处理；
- noisy→target：留出混音的SI-SDR应有正增益。

冒烟版只打印诊断；正式版若干净透传中位SI-SDR低于20 dB或去噪增益不为正，
直接停止，不把失败模型导出成“最终模型”。


In [ ]:
import numpy as np
import torch
from rtse.audio.io import read_audio
from rtse.data.dataset import stft_torch, istft_torch
from rtse.metrics.intrusive import si_sdr

@torch.inference_mode()
def run_wave(model, wave):
    x = torch.as_tensor(wave, dtype=torch.float32, device=next(model.parameters()).device)[None]
    spec = stft_torch(x)
    out_spec = model(spec)
    return istft_torch(out_spec, length=x.shape[-1])[0].cpu().numpy()

gate_results = {}
for name in MODELS:
    model = build_model(name).to('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(f'{CKPT_DIR}/{name}/best.pt', map_location=next(model.parameters()).device,
                    weights_only=False)
    model.load_state_dict(ck['model'])
    model.eval()

    clean_scores = []
    for path in manifest['speech']['val'][:10]:
        wave = read_audio(path)
        if wave.size < 16000:
            continue
        wave = wave[:16000 * 4]
        clean_scores.append(si_sdr(wave, run_wave(model, wave)))

    gains = []
    for i in range(10):
        noisy, target = val_ds[i]
        noisy_np, target_np = noisy.numpy(), target.numpy()
        gains.append(si_sdr(target_np, run_wave(model, noisy_np)) - si_sdr(target_np, noisy_np))

    clean_median = float(np.median(clean_scores))
    gain_median = float(np.median(gains))
    gate_results[name] = {'clean_passthrough_si_sdr': clean_median,
                          'noisy_delta_si_sdr': gain_median}
    print(f'{name}: clean透传 {clean_median:.2f} dB；留出混音 ΔSI-SDR {gain_median:+.2f} dB')

    if not SMOKE_RUN:
        assert clean_median >= 20.0, f'{name} 对干净输入仍然过抑制，禁止导出'
        assert gain_median > 0.0, f'{name} 留出混音没有正增益，禁止导出'

Path(f'{DRIVE}/training_gates.json').write_text(
    json.dumps(gate_results, ensure_ascii=False, indent=1), encoding='utf-8')


## 4. 训练曲线


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in MODELS:
    h = Path(f'{CKPT_DIR}/{name}/history.json')
    if not h.exists(): continue
    hist = json.loads(h.read_text())
    ep = [r['epoch'] for r in hist]
    axes[0].plot(ep, [r.get('loss') for r in hist], label=name)
    axes[1].plot(ep, [r.get('si_sdr') for r in hist], label=f'{name} train')
    if 'val_si_sdr' in hist[0]:
        axes[1].plot(ep, [r.get('val_si_sdr') for r in hist], '--', label=f'{name} val')
    axes[2].plot(ep, [r.get('spec') for r in hist], label=name)

for ax, t in zip(axes, ['总损失', 'SI-SDR (dB)', '压缩谱损失']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 怎么读：
#   训练 SI-SDR 一路涨但验证走平 → 过拟合，加数据或加正则
#   两条都走平且数值低         → 欠拟合，或学习率有问题